# Causal Cafe Lab — Growth Intelligence Team
Facilitator builds the interface; you build the causal decision logic.

> Prediction estimates what will happen. Causality estimates what an action
> will change. Policy decides what we should do.

**Running in Google Colab?** Run the setup cell below first. It downloads the
workshop files into this Colab session. Everything else runs unchanged.

In [ ]:
# Setup: fetch the workshop files when they are not already next to this notebook.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/tjunjie1408/cafe_lab.git"

if not Path("src/workshop_data.py").exists():
    if not Path("cafe_lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, "cafe_lab"],
            check=True,
        )
    os.chdir("cafe_lab")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))  # so `import src` finds the downloaded helpers
print("Working directory:", Path.cwd())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from src.workshop_data import (
    FEATURES, ARTIFACT_COLUMNS, GROWTH_BUDGET,
    load_observed_data,
)
from src.predictive import fit_purchase_baseline
from src.visualisation import (
    plot_causal_dag,
    plot_t_learner_schematic,
    plot_uplift_archetypes,
    plot_uplift_distribution,
    plot_workshop_roadmap,
)

observed = load_observed_data()
observed.head()

In [ ]:
plot_workshop_roadmap()

The old AI ranks customers by predicted purchase probability. High reported
ROI made the campaign look successful — until the audit.

In [ ]:
observed["purchase_probability"] = fit_purchase_baseline(observed)
observed[["customer_id", "purchase_probability", "coupon_sent", "purchased_7d"]].head()

A high purchase probability tells you a customer buys. It does not tell you
whether the coupon is the reason. Which of these four boxes does the old
model fill its target list from?

In [ ]:
plot_uplift_archetypes()

## Task 1 — Specify the question (worksheet)
In the cell below, write your answers as plain text:

- treatment
- outcome
- three plausible confounders
- one variable you must NOT control for because it happens after treatment
- a one-line DAG sketch
- the supplied funding rule (headquarters Growth Budget)

_Your answer here._

### Check your DAG (run after the group discussion)

In [ ]:
plot_causal_dag()

## Task 2 — Naive observed difference
Compute E[Y|T=1] - E[Y|T=0]. This is what we observed, not what we want.
It must NOT be labeled ATE.

In [ ]:
def naive_observed_difference(df: pd.DataFrame) -> float:
    # TODO: return treated purchase rate minus control purchase rate.
    raise NotImplementedError


naive = naive_observed_difference(observed)
print("naive observed difference:", naive)

## Task 3 — T-Learner
Split by coupon_sent, fit one outcome model per arm on FEATURES, then predict
mu0 and mu1 for every customer and take uplift = mu1 - mu0.

In [ ]:
plot_t_learner_schematic()

In [ ]:
def t_learner_uplift(df: pd.DataFrame, features=FEATURES):
    # TODO: fit two LogisticRegression models and return (mu0, mu1, uplift).
    raise NotImplementedError


mu0, mu1, uplift = t_learner_uplift(observed)
plot_uplift_distribution(uplift)

### Stretch (finished early?) — break the DAG on purpose
Your function takes a `features` argument. Re-run it without some confounders
and compare the mean uplift with the naive difference from Task 2:

1. drop only `loyalty_score`;
2. drop all five customer-history features (`days_since_last_purchase`,
   `orders_30d`, `app_sessions_30d`, `average_order_value`, `loyalty_score`).

What happens in each case, and why is the first one so mild?

## Task 4 — Campaign policy and decision memo
expected_incremental_profit = uplift * expected_order_margin - expected_coupon_cost

Recommend positive-value customers under the fixed GROWTH_BUDGET, then write
artifacts/customer_scores.csv with columns ARTIFACT_COLUMNS.

In [ ]:
def build_customer_scores(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: assemble purchase_probability, estimated_uplift,
    # expected_incremental_profit, recommended; enforce the budget.
    raise NotImplementedError


scores = build_customer_scores(observed)
Path("artifacts").mkdir(parents=True, exist_ok=True)
scores.to_csv("artifacts/customer_scores.csv", index=False)
scores.head()

### Check your policy
These are the numbers your decision memo needs. The facilitator will show how
the policy performs against the simulator's truth on the projector.

In [ ]:
chosen = scores["recommended"] == 1
spend = observed.loc[chosen, "expected_coupon_cost"].sum()
print(f"customers recommended:        {chosen.sum():,}")
print(f"expected coupon spend:        {spend:,.2f} / {GROWTH_BUDGET:,.0f}")
print(f"expected incremental orders:  {scores.loc[chosen, 'estimated_uplift'].sum():,.1f}")
print(f"expected incremental profit:  {scores.loc[chosen, 'expected_incremental_profit'].sum():,.2f}")
print(f"mean estimated uplift (ATE):  {scores['estimated_uplift'].mean():+.3f}")

## Reveal — watch the projector
The simulator knows every customer's true effect, which real data never does.
The facilitator will compare the estimate and the targeting policy with that
truth on the shared dashboard.

### Decision memo (fill in)
Estimated effect · identification assumptions · budget impact ·
recommended action · what still needs validation (e.g. a holdout experiment).